# Graphs — Subtopic 4: Multi-source BFS & Distance Problems

**Kernel:** C++17 (xeus-cling, `xcpp17`)

Pedagogical order — cleanest baseline first, then variants that stress one new idea each:
1. Distance of Nearest Cell Having 1 / 0-1 Matrix (multi-source BFS from all 1s — the clean baseline)
2. Rotten Oranges (multi-source BFS, deep dive: why a naive per-source loop is asymptotically worse, and why enqueueing all sources at level 0 is what makes simultaneous distances correct)
3. Shortest Distance in a Binary Maze (single-source BFS on a grid — contrast case, one source only)


## 4.1 Distance of Nearest Cell Having 1 (0-1 Matrix)

### State definition
Given a binary matrix, for every cell $(r,c)$ compute $\text{dist}[r][c]$ = the minimum 4-directional
step count to the **nearest** cell containing a 1 (cells that are themselves 1 have distance 0).

### Invariant — the multi-source layer invariant
This is Subtopic 2.1's BFS layer invariant, generalized: instead of one source, **every 1-cell is
simultaneously a layer-0 source**. The invariant becomes: at the moment cell $u$ is dequeued, every
cell $v$ with $\text{dist}[v] < \text{dist}[u]$ has already been dequeued — where distance is now
measured to the *nearest* member of the source set $S = \{(r,c) : \text{grid}[r][c] = 1\}$, not to
a single fixed source.

$$
\text{dist}[v] = \min_{s \in S} \big(\text{grid-distance}(s, v)\big), \qquad
S = \{(r,c) : \text{grid}[r][c] = 1\}
$$

### Base / initial conditions and why
- Every $s \in S$ gets $\text{dist}[s] = 0$ **and** is pushed into the queue **before the BFS main
  loop starts** — this is the generalization of "push source before entering loop" from Subtopic
  2.1, now applied to an entire set of sources instead of one.
- All non-source cells start at $\infty$ (unvisited), exactly as before.

### Why it works — proof that seeding ALL sources at layer 0 gives correct nearest-distance

**Claim:** BFS seeded with every $s \in S$ at distance 0 simultaneously computes
$\text{dist}[v] = \min_{s \in S} \text{grid-distance}(s,v)$ for every $v$.

*Proof:* This is a direct corollary of Subtopic 2.1's single-source proof, applied to an
**augmented graph** $G'$: add a virtual super-source $\sigma$ with a zero-cost edge to every
$s \in S$. A BFS on $G'$ from $\sigma$ gives $\text{dist}_{G'}[\sigma][v] = 1 + \min_{s \in S}
\text{grid-distance}(s,v)$ (by the definition of BFS distance and the single hop from $\sigma$ to
its nearest useful $s$). Practically implementing $\sigma$ as "seed every $s \in S$ at distance 0
and skip the +1 hop" is exactly equivalent — it is the standard trick of collapsing the virtual
super-source into "push all sources at once, all starting at 0." Since Subtopic 2.1 already proved
single-source BFS from $\sigma$ is correct, and this collapsing is a pure notational simplification
(the +1 hop for reaching $\sigma$ never actually needs computing since it's identical for every
source), the multi-source result $\text{dist}[v] = \min_{s} \text{grid-distance}(s,v)$ follows
directly. $\blacksquare$

### Boundary transitions table

| Decision point | Condition | Action |
|---|---|---|
| Cell is a 1 | `grid[r][c] == 1` | seed at `dist=0`, push to queue before main loop |
| Cell is a 0, unvisited | `dist[r][c] == -1` (or ∞) | discovered via BFS, `dist = dist[parent] + 1` |
| All-0 grid | $S = \emptyset$ | no sources to seed — every distance is undefined/unreachable, must special-case |

### The delta
The non-obvious insight: multi-source BFS is **not** "run single-source BFS once per source and
take the min" — that would be correct but wasteful (see 4.2's deep dive on exactly this
comparison). It is a **single** BFS run whose queue starts pre-loaded with every source at layer 0
simultaneously — the layer invariant then does the "take the min over all sources" work for free,
because whichever source's frontier reaches a cell first is, by the layer invariant, the nearest
one.


In [ ]:
// Distance of Nearest Cell Having 1: multi-source BFS, all 1-cells seeded at layer 0
#include <iostream>
#include <vector>
#include <queue>
using namespace std;
#define vi vector<int>
#define vvi vector<vector<int>>

vvi nearestOne(vvi& grid) {
    int R = grid.size(), C = grid[0].size();
    vvi dist(R, vi(C, -1));           // -1 = unvisited/infinity
    queue<pair<int,int>> q;

    // Seed step: EVERY 1-cell is a layer-0 source, pushed before the main loop starts.
    for (int r = 0; r < R; r++)
        for (int c = 0; c < C; c++)
            if (grid[r][c] == 1) { dist[r][c] = 0; q.push({r, c}); }

    int dr[4] = {-1,1,0,0}, dc[4] = {0,0,-1,1};
    while (!q.empty()) {2
        auto [r, c] = q.front(); q.pop();
        for (int d = 0; d < 4; d++) {
            int nr = r+dr[d], nc = c+dc[d];
            if (nr < 0 || nr >= R || nc < 0 || nc >= C) continue;
            if (dist[nr][nc] != -1) continue;          // already reached by some (possibly other) source
            dist[nr][nc] = dist[r][c] + 1;              // one step farther than the frontier cell that found it
            q.push({nr, nc});
        }
    }
    return dist;
}


In [ ]:
// Test cell: input -> actual (Expected: X)

// Single 1 in a corner
{
    vvi g = {{1,0},{0,0}};
    vvi d = nearestOne(g);
    cout << "nearestOne corner, d[1][1] -> " << d[1][1] << " (Expected: 2)\n";
}

// Two 1s -- nearest should take the min over both sources
{
    vvi g = {{1,0,0,1},{0,0,0,0}};
    vvi d = nearestOne(g);
    cout << "nearestOne two sources, d[0][1] -> " << d[0][1] << " d[0][2] -> " << d[0][2]
         << " (Expected: 1, 1 -- each nearest to its own closer 1)\n";
}

// All 1s -- every distance is 0
{
    vvi g = {{1,1},{1,1}};
    vvi d = nearestOne(g);
    cout << "nearestOne all ones, d[1][1] -> " << d[1][1] << " (Expected: 0)\n";
}

// Single cell, itself a 1
{
    vvi g = {{1}};
    vvi d = nearestOne(g);
    cout << "nearestOne single cell (is 1) -> " << d[0][0] << " (Expected: 0)\n";
}

// Sparse grid: one 1 far from a large empty region
{
    vvi g = {{0,0,0,0,0},{0,0,0,0,0},{0,0,1,0,0},{0,0,0,0,0},{0,0,0,0,0}};
    vvi d = nearestOne(g);
    cout << "nearestOne sparse, d[0][0] -> " << d[0][0] << " (Expected: 4, Manhattan distance to center)\n";
}


## 4.2 Rotten Oranges (Multi-Source BFS — Deep Dive)

### State definition
A grid has `0` (empty), `1` (fresh orange), `2` (rotten orange). Every minute, every rotten orange
rots its 4-directional fresh neighbors. Find the minimum minutes until no fresh orange remains, or
$-1$ if impossible.

### Invariant
Identical multi-source layer invariant to 4.1 — every initially-rotten orange is a layer-0 source,
and $\text{dist}[r][c]$ (minutes until rotten) is exactly the BFS layer of $(r,c)$. The answer is
$\max_{(r,c) \text{ fresh}} \text{dist}[r][c]$ (the last fresh orange to rot determines total time),
or $-1$ if some fresh orange is never reached at all.

### Deep dive — why a naive single-source-per-rotten-orange loop is asymptotically worse

**Naive approach:** for each rotten orange independently, run a full single-source BFS over the
grid, and take the per-cell minimum across all these runs.

**Cost of the naive approach:** if there are $k$ initially-rotten oranges, this runs $k$ separate
BFS passes, each $O(R \cdot C)$, for a total of $O(k \cdot R \cdot C)$. In the worst case $k =
\Theta(R \cdot C)$ (up to half the grid is rotten already), giving $O((R \cdot C)^2)$ — **quadratic**
in grid size.

**Cost of multi-source BFS:** a single BFS pass, seeded with all $k$ rotten oranges simultaneously
at layer 0, is $O(R \cdot C)$ regardless of $k$ — each cell is dequeued and its neighbors examined
exactly once, by the visited-set discipline (Subtopic 2.5).

**Why simultaneous seeding gives the exact right simulation, not just a shortcut:** the problem's
real-world semantics — *all* rotten oranges rot their neighbors *in the same minute*, in parallel —
is *exactly* what "push every source at layer 0 together" computes. If you instead ran the BFS
sources one at a time (even if you then took a min at the end, which — as shown in 4.1's proof —
gives the mathematically correct distances), you would still get the right final **distance
values**, but at $O(k \cdot R \cdot C)$ cost, wastefully re-deriving what one simultaneous pass
gets for free. This is the concrete, quantified version of 4.1's delta: multi-source BFS isn't just
notationally cleaner, it is a genuine $\Theta(k)$-factor speedup whenever $k$ is large.

### Why it works — formal correctness
By 4.1's proof (the virtual-super-source argument), the multi-source BFS layer of a fresh orange
equals its true minimum rotting time — since every rotten orange is symmetric as an infection
source (any one of them rotting a cell first is what matters, exactly matching $\min_{s \in S}$).
The final answer, $\max$ over all fresh-orange layers, is correct because *all* fresh oranges must
be rotten for the process to finish, and the process finishes exactly when the slowest-to-rot one
is reached.

### Boundary transitions table

| Decision point | Condition | Action |
|---|---|---|
| Initially rotten | `grid[r][c] == 2` | seed at layer 0, push before main loop |
| Fresh orange reached | `grid[r][c] == 1`, unvisited | mark rotten, `dist = dist[parent]+1`, push |
| No fresh oranges at all | count of 1s is 0 initially | answer is 0 immediately (nothing to wait for) |
| Fresh orange never reached | still `1` (unvisited) after BFS completes | answer is $-1$ — impossible |

### The delta
The non-obvious insight, sharpened from 4.1: the *reason* multi-source BFS is correct here is not
just mathematical convenience — it is a **faithful model of true parallel/simultaneous spreading**.
Any problem phrased as "all X simultaneously affect their neighbors each time step, find when
everything is affected" (fire spread, infection spread, gas diffusion) is a multi-source BFS in
disguise, and the naive per-source-loop approach, while not *incorrect*, throws away a real
asymptotic factor for no benefit.


In [ ]:
// Rotten Oranges: multi-source BFS, minutes = BFS layer, answer = max layer over fresh oranges
#include <iostream>
#include <vector>
#include <queue>
using namespace std;
#define vi vector<int>
#define vvi vector<vector<int>>

int orangesRotting(vvi grid) {
    int R = grid.size(), C = grid[0].size();
    queue<pair<int,int>> q;
    int freshCount = 0;

    // Seed step: every initially-rotten orange is a layer-0 source, ALL pushed before the main loop.
    for (int r = 0; r < R; r++) {
        for (int c = 0; c < C; c++) {
            if (grid[r][c] == 2) q.push({r, c});
            else if (grid[r][c] == 1) freshCount++;
        }
    }

    if (freshCount == 0) return 0;   // base case: nothing fresh, no time needs to pass

    int minutes = 0;
    int dr[4] = {-1,1,0,0}, dc[4] = {0,0,-1,1};

    // Standard multi-source BFS, processed level-by-level so `minutes` tracks the true layer count.
    while (!q.empty()) {
        int sz = q.size();
        bool rottedThisRound = false;
        for (int i = 0; i < sz; i++) {
            auto [r, c] = q.front(); q.pop();
            for (int d = 0; d < 4; d++) {
                int nr = r+dr[d], nc = c+dc[d];
                if (nr < 0 || nr >= R || nc < 0 || nc >= C) continue;
                if (grid[nr][nc] != 1) continue;           // not fresh: empty or already rotten
                grid[nr][nc] = 2;                           // rot it -- also serves as visited marker
                freshCount--;
                rottedThisRound = true;
                q.push({nr, nc});
            }
        }
        if (rottedThisRound) minutes++;                    // only advance the clock if something changed
    }

    return freshCount == 0 ? minutes : -1;                  // unreached fresh oranges -> impossible
}


In [ ]:
// Test cell: input -> actual (Expected: X)

// Standard case: rot spreads from one corner
{
    vvi g = {{2,1,1},{1,1,0},{0,1,1}};
    cout << "orangesRotting(standard) -> " << orangesRotting(g) << " (Expected: 4)\n";
}

// Impossible: isolated fresh orange, unreachable
{
    vvi g = {{2,1,1},{0,1,1},{1,0,2}};
    cout << "orangesRotting(isolated fresh) -> " << orangesRotting(g) << " (Expected: -1)\n";
}

// No fresh oranges at all -> 0 minutes needed
{
    vvi g = {{0,2}};
    cout << "orangesRotting(no fresh) -> " << orangesRotting(g) << " (Expected: 0)\n";
}

// Empty grid entirely (all zeros) -> 0 minutes, nothing fresh
{
    vvi g = {{0,0},{0,0}};
    cout << "orangesRotting(all empty) -> " << orangesRotting(g) << " (Expected: 0)\n";
}

// Single fresh orange, no rotten source anywhere -> impossible
{
    vvi g = {{1}};
    cout << "orangesRotting(single fresh, no source) -> " << orangesRotting(g) << " (Expected: -1)\n";
}

// Multiple rotten sources speeding up the spread (multi-source advantage, contrast with single-source math)
{
    vvi g = {{2,1,1,1,2}};
    cout << "orangesRotting(two sources from both ends) -> " << orangesRotting(g) << " (Expected: 2, meets in the middle twice as fast)\n";
}


## 4.3 Shortest Distance in a Binary Maze (Single-Source BFS — Contrast Case)

### State definition
Given a binary maze (`0` = wall, `1` = open path), find the shortest 4-directional path length from
a single given source $(sr,sc)$ to a single given destination $(dr,dc)$, or $-1$ if unreachable.

### Invariant
This is the **plain single-source** BFS layer invariant from Subtopic 2.1, applied directly to a
grid — exactly one source, no multi-seeding. It is included here deliberately as a contrast case:
after 4.1 and 4.2, it is tempting to reach for multi-source machinery reflexively; this problem is
the reminder that multi-source BFS is a **generalization**, not a universal replacement — use it
only when the problem genuinely has multiple simultaneous sources.

### Why it works
Direct instantiation of Subtopic 2.1's proof: $\text{dist}[dr][dc]$, once the destination is
dequeued, equals the true shortest number of 4-directional steps from $(sr,sc)$, since grid
adjacency is a valid unweighted graph and BFS's non-decreasing-distance dequeue order guarantees
minimality.

### Boundary transitions table

| Decision point | Condition | Action |
|---|---|---|
| Source or destination is a wall | `maze[sr][sc]==0` or `maze[dr][dc]==0` | immediately return $-1$, no valid path can exist |
| Source equals destination | $(sr,sc) = (dr,dc)$ | return 0 — zero steps needed |
| Cell is open, unvisited | `maze[r][c]==1`, `dist==-1` | discover, `dist = dist[parent]+1` |
| Destination dequeued | `(r,c) == (dr,dc)` | can early-exit — nothing further can improve the answer (layer invariant) |

### The delta
The non-obvious insight, closing the loop on this subtopic: the difference between 4.1/4.2 and 4.3
is entirely in the **seed step** — one source vs. many. Every other part of the algorithm (bounds
checking, visited discipline, layer-by-layer expansion) is identical. Recognizing "how many
simultaneous sources does this problem actually have" is the single decision that determines which
of these two patterns to reach for — get it wrong (single-seeding a problem that has many true
sources) and you pay the same quadratic penalty proven in 4.2's deep dive.


In [ ]:
// Shortest Distance in a Binary Maze: single-source BFS, contrast to 4.1/4.2's multi-source pattern
#include <iostream>
#include <vector>
#include <queue>
using namespace std;
#define vi vector<int>
#define vvi vector<vector<int>>

int shortestPathBinaryMaze(vvi& maze, pair<int,int> src, pair<int,int> dest) {
    int R = maze.size(), C = maze[0].size();
    auto [sr, sc] = src;
    auto [dr, dc] = dest;

    if (maze[sr][sc] == 0 || maze[dr][dc] == 0) return -1;  // wall at source/destination: no path possible
    if (sr == dr && sc == dc) return 0;                      // already there: zero steps

    vvi dist(R, vi(C, -1));
    queue<pair<int,int>> q;
    dist[sr][sc] = 0;         // single source seeded at layer 0 -- ONE source, not many
    q.push({sr, sc});

    int drr[4] = {-1,1,0,0}, dcc[4] = {0,0,-1,1};
    while (!q.empty()) {
        auto [r, c] = q.front(); q.pop();
        if (r == dr && c == dc) return dist[r][c];   // early exit: layer invariant guarantees this is optimal
        for (int d = 0; d < 4; d++) {
            int nr = r+drr[d], nc = c+dcc[d];
            if (nr < 0 || nr >= R || nc < 0 || nc >= C) continue;
            if (maze[nr][nc] == 0 || dist[nr][nc] != -1) continue;  // wall or already-visited: skip
            dist[nr][nc] = dist[r][c] + 1;
            q.push({nr, nc});
        }
    }
    return -1;  // destination never reached
}


In [ ]:
// Test cell: input -> actual (Expected: X)

// Simple open path
{
    vvi m = {{1,1,1},{0,0,1},{1,1,1}};
    int d = shortestPathBinaryMaze(m, {0,0}, {2,0});
    cout << "shortestPathBinaryMaze(open path) -> " << d << " (Expected: 6, forced around the wall column)\n";
}

// Source equals destination
{
    vvi m = {{1,1},{1,1}};
    int d = shortestPathBinaryMaze(m, {0,0}, {0,0});
    cout << "shortestPathBinaryMaze(src==dest) -> " << d << " (Expected: 0)\n";
}

// Destination is a wall -> impossible
{
    vvi m = {{1,1},{1,0}};
    int d = shortestPathBinaryMaze(m, {0,0}, {1,1});
    cout << "shortestPathBinaryMaze(dest is wall) -> " << d << " (Expected: -1)\n";
}

// Fully blocked maze between src and dest
{
    vvi m = {{1,0,1},{0,0,0},{1,0,1}};
    int d = shortestPathBinaryMaze(m, {0,0}, {2,2});
    cout << "shortestPathBinaryMaze(fully blocked) -> " << d << " (Expected: -1)\n";
}

// Single cell maze, src == dest
{
    vvi m = {{1}};
    int d = shortestPathBinaryMaze(m, {0,0}, {0,0});
    cout << "shortestPathBinaryMaze(single cell) -> " << d << " (Expected: 0)\n";
}

// Dense open grid (all 1s), straight-line Manhattan distance expected
{
    vvi m(4, vi(4, 1));
    int d = shortestPathBinaryMaze(m, {0,0}, {3,3});
    cout << "shortestPathBinaryMaze(open 4x4 grid) -> " << d << " (Expected: 6, Manhattan distance)\n";
}


In [ ]:
// ================= UNIFIED MENTAL MODEL — Subtopic 4 =================
#include <iostream>
#include <vector>
#include <queue>
using namespace std;
#define vi vector<int>
#define vvi vector<vector<int>>

// --- Single-source BFS template (Subtopic 2.1, reused unchanged) ---
vvi singleSourceBFS(vvi& grid, int passValue, pair<int,int> src) {
    int R = grid.size(), C = grid[0].size();
    vvi dist(R, vi(C, -1));
    queue<pair<int,int>> q;
    dist[src.first][src.second] = 0;
    q.push(src);
    int dr[4]={-1,1,0,0}, dc[4]={0,0,-1,1};
    while (!q.empty()) {
        auto [r,c] = q.front(); q.pop();
        for (int d=0; d<4; d++) {
            int nr=r+dr[d], nc=c+dc[d];
            if (nr<0||nr>=R||nc<0||nc>=C) continue;
            if (grid[nr][nc] != passValue || dist[nr][nc] != -1) continue;
            dist[nr][nc] = dist[r][c] + 1;
            q.push({nr,nc});
        }
    }
    return dist;
}

// --- Multi-source BFS template (Subtopic 4's generalization) ---
// The ONLY structural difference from single-source: the seed step loops over ALL sources
// and pushes them together before the main loop begins.
vvi multiSourceBFS(vvi& grid, int passValue, vector<pair<int,int>>& sources) {
    int R = grid.size(), C = grid[0].size();
    vvi dist(R, vi(C, -1));
    queue<pair<int,int>> q;
    for (auto& s : sources) { dist[s.first][s.second] = 0; q.push(s); }   // <-- the one changed line
    int dr[4]={-1,1,0,0}, dc[4]={0,0,-1,1};
    while (!q.empty()) {
        auto [r,c] = q.front(); q.pop();
        for (int d=0; d<4; d++) {
            int nr=r+dr[d], nc=c+dc[d];
            if (nr<0||nr>=R||nc<0||nc>=C) continue;
            if (grid[nr][nc] != passValue || dist[nr][nc] != -1) continue;
            dist[nr][nc] = dist[r][c] + 1;
            q.push({nr,nc});
        }
    }
    return dist;
}

int main() {
    vvi grid = {{0,1,1},{1,1,0},{1,0,0}};   // pass value 1
    vector<pair<int,int>> sources = {{0,1},{2,0}};
    vvi d = multiSourceBFS(grid, 1, sources);
    cout << "Unified template sanity check: d[1][1] -> " << d[1][1] << " (Expected: 1, nearest of the two sources)\n";
    return 0;
}


### Decision Tree — Subtopic 4

```
How many cells count as a valid "distance-0" starting point?
│
├── EXACTLY ONE (a single given source/start cell)
│    → Single-source BFS (Subtopic 2.1 template, unchanged) — e.g. Shortest Distance in Binary Maze (4.3)
│
└── MORE THAN ONE, and they all act simultaneously / in parallel
     → Multi-source BFS: seed ALL sources at layer 0 together, before the main loop
       │
       Does the problem ask for "distance to nearest source" per cell?
       ├── YES → 4.1 pattern (Distance of Nearest Cell Having 1 / 0-1 Matrix)
       │
       Does the problem simulate a process spreading over discrete time steps, and ask
       "how long until everything is affected / is it even possible"?
       ├── YES → 4.2 pattern (Rotten Oranges) — process level-by-level, track elapsed layers,
       │         answer = max layer reached (or -1 if some target cell is never reached)


Tempted to loop "run single-source BFS once per source, take the min"?
→ Mathematically correct (proved equivalent in 4.1) but costs O(k * R * C) instead of O(R * C)
  for k sources — ALWAYS prefer the simultaneous multi-source seed when k can be large (4.2 deep dive).
```


### Complexity Summary — Subtopic 4

| Pattern | Time | Space | When to use | Key invariant | Failure mode if misused |
|---|---|---|---|---|---|
| Single-source BFS (grid) | $O(R \cdot C)$ | $O(R \cdot C)$ | Exactly one true starting point | non-decreasing distance dequeue order from the one source | using it when the problem actually has multiple simultaneous sources under-counts spread speed |
| Multi-source BFS, nearest-distance (4.1) | $O(R \cdot C)$ | $O(R \cdot C)$ | Need distance-to-nearest-source for every cell | all sources seeded at layer 0 = virtual super-source BFS | looping single-source BFS per source instead: correct but $O(k \cdot R \cdot C)$, quadratic when $k = \Theta(RC)$ |
| Multi-source BFS, time-to-cover (4.2) | $O(R \cdot C)$ | $O(R \cdot C)$ (or $O(1)$ extra if grid reused as marker) | Simultaneous spreading process, need total time / feasibility | BFS layer = elapsed time step; answer = max layer among targets, or $-1$ if any target unreached | forgetting to check "unreached target remains" and returning `minutes` unconditionally gives a wrong answer on impossible cases |
